# Feature Engineering

Some features have already been engineered as part of the process of merging the eaglei, NOAA, and county-level predictors

We'll start by loading the data, creating new columns that will hold engineered values, and imputing some missing data from water-adjacent counties

Then, we'll engineer features on the merged dataset. For each fips code, we'll compute
- The mean value of each ERA5 predictor from all the neighboring counties
- The max value of each ERA5 predictor from all neighboring counties
- The total 12-hour and 24-hour precipitation and snowfall within each county

In [ ]:
import pandas as pd

## Loading and Cleaning

We'll start by loading the full data set and converting fips_code into an integer and neighbors into a list of (integer) fips_code values. This will support some merging/functions below.

In [ ]:
#Load the ../Data/Merged_Data/eaglei_noaa_era5_full.parquet file
df = pd.read_parquet('../Data/Merged_Data/eaglei_noaa_era5_full.parquet')

#Convert fips_code to a float, round it to the nearest integer, and then convert to an int64
df['fips_code'] = df['fips_code'].astype(float).round().astype('int64')

#Use json to convert each value of neighbors into a list of integers
import json
df['neighbors'] = df['neighbors'].apply(lambda x: json.loads(x))

In [ ]:
#Create a new dataframe
df_era5_grouped = pd.DataFrame()

#iterate through fips_code values
for fips_code in df['fips_code'].unique():

    #get the neighbors list for the current fips_code
    neighbors_list = df.loc[df['fips_code'] == fips_code, 'neighbors'].values[0]

    #make a new dataframe where fips_code is in neighbors_list
    df_neighbors = df[df['fips_code'].isin(neighbors_list)] 

    #group df_neighbors by datetime, and compute the mean and max of t2m, u10, v10, sf, and tp, ignoring NaN values
    df_neighbors_grouped = df_neighbors.groupby('datetime').agg({'t2m': ['mean', 'max'], 'u10': ['mean', 'max'], 'v10': ['mean', 'max'], 'sf': ['mean', 'max'], 'tp': ['mean', 'max']})

    #Make the grouped aggregated data into columns
    df_neighbors_grouped.columns = ['_'.join(col).strip() for col in df_neighbors_grouped.columns.values]

    #Add fips_code as a column
    df_neighbors_grouped['fips_code'] = fips_code

    #concatenate df_neighbors_grouped with df_era5_grouped
    df_era5_grouped = pd.concat([df_era5_grouped, df_neighbors_grouped], axis=0)

df_era5_grouped = df_era5_grouped.reset_index()

#Merge df_era5_grouped with df
df = df.merge(df_era5_grouped, on=['datetime', 'fips_code'], how='left')


## Imputing missing ERA5 data

Several ocean-adjacent counties are missing ERA5 data:
- 12087: Monroe, FL; centroid is (-81.1 25.3) and neighbors are [12021, 12086]
- 25019: Nantucket, MA; centroid is (-70.1 41.3) but there are no neighbors
- 34017: Hudson, NJ; centroid is (-74.1 40.7) and neighbors are [36061, 34003, 34013]
- 37031: Carteret, NC; centroid is (-76.7 34.8) and neighbors are [37133, 37103, 37049]
- 48007: Aransas, TX; centroid is (-97 28.1) and neighbors are [48355, 48057, 48409, 48391]
- 51115: Mathews, VA; centroid is (-76.3 37.4) and neighbors are [51119, 51073]
- 51810: Virginia Beach, VA; centroid is (-76 36.7) and neighbors are [37053, 51710, 51550]
- 53029: Island, WA; centroid is (-122.5 48.2) and neighbors are [53061]

We can now impute these using the mean of the values from their neighboring counties (except for Nantucket... which we should probably just drop)

In [21]:
#For the following values of fips_code, 
# replace NaN values of t2m, u10, v10, sf, and tp 
# with corresponding values of t2m_mean, u10_mean, v10_mean, sf_mean, and tp_mean
# 12087, 34017, 37031, 48007, 51115, 51810, and 53029
fips_codes = [12087, 34017, 37031, 48007, 51115, 51810, 53029]
for fips_code in fips_codes:
    df.loc[(df['fips_code'] == fips_code) & (df['t2m'].isna()), 't2m'] = df.loc[(df['fips_code'] == fips_code) & (df['t2m'].isna()), 't2m_mean']
    df.loc[(df['fips_code'] == fips_code) & (df['u10'].isna()), 'u10'] = df.loc[(df['fips_code'] == fips_code) & (df['u10'].isna()), 'u10_mean']
    df.loc[(df['fips_code'] == fips_code) & (df['v10'].isna()), 'v10'] = df.loc[(df['fips_code'] == fips_code) & (df['v10'].isna()), 'v10_mean']
    df.loc[(df['fips_code'] == fips_code) & (df['sf'].isna()), 'sf'] = df.loc[(df['fips_code'] == fips_code) & (df['sf'].isna()), 'sf_mean']
    df.loc[(df['fips_code'] == fips_code) & (df['tp'].isna()), 'tp'] = df.loc[(df['fips_code'] == fips_code) & (df['tp'].isna()), 'tp_mean']

## Computing weather features

- Wind speed (from u and v components)
- 12- and 24-hour total precipitation and snowfall

In [26]:
#Create a new column 'wind_speed' which is the square root of the sum of the squares of the 'u' and 'v' columns
df['wind_speed'] = (df['u10']**2 + df['v10']**2)**0.5
df['neighbor_mean_wind_speed'] = (df['u10_mean']**2 + df['v10_mean']**2)**0.5
df['neighbor_max_wind_speed'] = (df['u10_max']**2 + df['v10_max']**2)**0.5

#Drop the 'u' and 'v' columns
df = df.drop(columns=['u10', 'v10', 'u10_mean', 'v10_mean', 'u10_max', 'v10_max'])

In [27]:
# Compute cumulative amounts of snowfall and precipitation over 12- and 24-hour intervals
df['sf_12h'] = df.groupby('fips_code')['sf'].shift(1) + df['sf']
df['sf_24h'] = df.groupby('fips_code')['sf'].shift(1) + df.groupby('fips_code')['sf'].shift(2) + df.groupby('fips_code')['sf'].shift(3) + df['sf']

df['tp_12h'] = df.groupby('fips_code')['tp'].shift(1) + df['tp']
df['tp_24h'] = df.groupby('fips_code')['tp'].shift(1) + df.groupby('fips_code')['tp'].shift(2) + df.groupby('fips_code')['tp'].shift(3) + df['tp']

In [28]:
#Export to a parquet file
df.to_parquet('../Data/Merged_Data/eaglei_noaa_era5_engineered.parquet', index=False)